In [2]:

!apt-get update && apt-get install -y zstd

!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama pymupdf sentence-transformers faiss-cpu

import subprocess
import time

print("\nUruchamianie serwera Ollama...")
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

print("Pobieranie modelu llama3.2 (to potrwa chwilę)...")
!ollama pull llama3.2
print("Gotowe! Środowisko przygotowane.")

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,026 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,221 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,307 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.4 MB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 k

In [3]:
import os
import json
import faiss
import pymupdf
import numpy as np
from tqdm import tqdm
import ollama
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings("ignore")


embedder = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')
index = faiss.IndexFlatL2(embedder.get_embedding_dimension())
metadata = []

class RAGSystem:
    def __init__(self, embedder, index, metadata, chunk_size=512):
        self.embedder = embedder
        self.index = index
        self.metadata = metadata
        self.chunk_size = chunk_size

    def process_file(self, file_path):
        text = []
        pdf_document = pymupdf.open(file_path)
        for page_num in range(len(pdf_document)):
            page = pdf_document.load_page(page_num)
            text.append((page_num, str(page.get_text()).replace("\n", " ")))

        chunks = []
        for page_num, page_text in text:
            page_chunks = [(page_num, page_text[i:i+self.chunk_size]) for i in range(0, len(page_text), self.chunk_size)]
            chunks.extend(page_chunks)

        for chunk_num, (page_number, chunk) in enumerate(tqdm(chunks, desc=f"Analiza: {os.path.basename(file_path)}")):
            embeddings = self.embedder.encode(chunk, show_progress_bar=False)
            self.index.add(np.array([embeddings]))
            self.metadata.append({
                "filename": os.path.basename(file_path),
                "page_number": page_number,
                "chunk": chunk
            })

    def answer_question(self, query):

        q_embedding = self.embedder.encode(query, show_progress_bar=False)
        D, I = self.index.search(np.array([q_embedding]), 5)
        chunks = [self.metadata[i] for i in I[0]]


        context = ""
        for i, chunk in enumerate(chunks):
            context += f"Dokument [{i+1}]: {chunk['chunk']}\n"


        messages = [
            {
                "role": "system",
                "content": (
                    "Jesteś precyzyjnym asystentem RAG. Odpowiadaj w języku polskim TYLKO na podstawie dostarczonego kontekstu. "
                    "Jeśli w kontekście brakuje informacji pozwalających na odpowiedź, napisz DOKŁADNIE to zdanie: "
                    "'Kontekst nie zawiera informacji pozwalających na odpowiedź.' "
                    "Kategorycznie zabraniam dodawania własnej wiedzy i zmyślania."
                )
            },
            {"role": "user", "content": f"Kontekst:\n{context}\n\nPytanie: {query}"}
        ]

        response = ollama.chat(model="llama3.2", messages=messages, options={"temperature": 0.1})
        return response['message']['content'], chunks

rag = RAGSystem(embedder, index, metadata)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/402 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
from IPython.display import display, Markdown

knowledge_dir = "knowledge/"

print("Pobieranie wiedzy z plików PDF...\n")
for file in os.listdir(knowledge_dir):
    if file.endswith('.pdf'):
        rag.process_file(os.path.join(knowledge_dir, file))

print(f"\nGotowe! Liczba wczytanych fragmentów w bazie: {index.ntotal}\n")

pytania = [
    "Jakie typy danych astronomicznych analizuje się za pomocą ML?",
    "Dlaczego wykrywanie małych egzoplanet jest trudniejsze niż dużych?",
    "Jaki jest przepis na dobre ciasto czekoladowe?"
]

for pyt in pytania:
    odpowiedz, zrodla = rag.answer_question(pyt)

    display(Markdown(f"### **Pytanie:** {pyt}"))
    display(Markdown(f"**Odpowiedź:**\n\n{odpowiedz}"))
    display(Markdown("**Źródła:**"))
    for i, chunk in enumerate(zrodla):
        fragment = chunk['chunk'][:150].strip() + "..."
        display(Markdown(f"**[{i+1}]** `{chunk['filename']}` — strona {chunk['page_number'] + 1}\n> _{fragment}_"))
    display(Markdown("<hr>"))

Pobieranie wiedzy z plików PDF...



Analiza: Astrochemistry_of_dust.pdf: 100%|██████████| 275/275 [01:37<00:00,  2.81it/s]



Gotowe! Liczba wczytanych fragmentów w bazie: 913



### **Pytanie:** Jakie typy danych astronomicznych analizuje się za pomocą ML?

**Odpowiedź:**

Dokument [4]: Wykorzystuje się regresję, szacowanie rozkładu, redukcję wymiarów, analizę cyklów i hierarchiczne grupowanie.

...

**[2]** `NASA_Space_Based_Astronomy.pdf` — strona 24
> _ss or Plexiglass TM Emery paper (fine) to smooth sharp edges of glass or plastic Demonstration 2 Shallow dish or pie tin Empty coffee can Ice Spray bo..._

**[3]** `Astrochemistry_of_dust.pdf` — strona 25
> _observations which can spatially resolve the hot cores and map the different molecules. A ‘sweet taste’ of what to expect is the recent detection of t..._

**[4]** `NASA_Space_Based_Astronomy.pdf` — strona 42
> _5. Place a 4 by 4-centimeter square of alu- minum foil on a cutting surface. Cut a nar- row slot into the foil with the razor blade knife. If you made..._

**[5]** `NASA_Space_Based_Astronomy.pdf` — strona 63
> _e knife Cutting surface Marker pen Rubber cement Fine grade sandpaper Concave makeup mirror Electric holiday candle or other small light source Dark r..._

<hr>